## part 1

In [1]:
pip install numpy scikit-learn matplotlib torch torchvision torchaudio pandas seaborn


[notice] A new release of pip is available: 26.2.1 -> 26.3
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [2]:
import numpy as np
import sklearn
import matplotlib.pyplot as plt
import torch
import pandas as pd
import seaborn as sns

In [4]:
import pandas as pd
from sklearn.model_selection import train_test_split

from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, r2_score

pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", None)


In [ ]:
# Import data
train_path = "outcome_train.csv"
val_path = "outcome_val.csv"
test_path = "outcome_test.csv"

df_train1 = pd.read_csv(train_path)
df_valid1 = pd.read_csv(val_path)
df_test1 = pd.read_csv(test_path)

# Separate dataset by trial

# For each phase (1-3) build binary classifier that predicts probability label=1
    # Choose features
    # Preprocessing
    # What to do with criteria?

In [ ]:
# Split training data by phase
# Combined-phase trials (e.g. phase 1/phase 2) have one row per phase, so they appear in both frames
df_train_I = df_train1[df_train1["phase"] == "I"].reset_index(drop=True)
df_train_II = df_train1[df_train1["phase"] == "II"].reset_index(drop=True)
df_train_III = df_train1[df_train1["phase"] == "III"].reset_index(drop=True)

print(df_train_I.shape, df_train_II.shape, df_train_III.shape)

## Part 2: predicting trial outcomes

One probability model serves both goals. Kaggle scores log loss on the probabilities, and F1 comes only from a per-phase decision threshold.

Each candidate is scored three ways:
- **Grouped CV on train:** folds are grouped by `trial_id`, so a combined-phase trial never sits on both sides. Thresholds are also chosen here.
- **`outcome_val.csv`:** the reported numbers.
- **Time holdout:** train on trials up to 2009 and score 2010-13, using train rows only. This is the closest stand-in for the later test years.

In [ ]:
import ast
import re
import warnings

from joblib import Parallel, delayed
from rdkit import Chem, RDLogger
from rdkit.Chem import Descriptors, rdFingerprintGenerator
from scipy import sparse
from scipy.special import expit, logit
from sklearn.decomposition import TruncatedSVD
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import average_precision_score, f1_score, log_loss, precision_score, recall_score, roc_auc_score
from sklearn.model_selection import GroupKFold
from sklearn.naive_bayes import MultinomialNB
from sklearn.neighbors import KNeighborsClassifier
from sklearn.preprocessing import OneHotEncoder, StandardScaler, normalize

warnings.filterwarnings("ignore", category=FutureWarning)
RDLogger.DisableLog("rdApp.*")

PHASES = ["I", "II", "III"]
N_JOBS = 8

y_train, y_val = df_train1["label"].values, df_valid1["label"].values
groups_train = df_train1["trial_id"].values
phase_train, phase_val = df_train1["phase"].values, df_valid1["phase"].values

# Sanity checks on the splits
assert not set(df_train1["trial_id"]) & set(df_valid1["trial_id"])
assert not set(df_train1["trial_id"]) & set(df_test1["trial_id"])
for frame in [df_train1, df_valid1, df_test1]:
    assert not frame.duplicated(["trial_id", "phase"]).any()

### Feature engineering

These are row-by-row transforms with nothing learned from the data. Vectorizers and scalers are fit later, inside each fold. `trial_id`, reason labels and reason embeddings are never used as features.

In [ ]:
fp_generator = rdFingerprintGenerator.GetMorganGenerator(radius=2, fpSize=1024)
catalog = pd.read_csv("pathway_catalog.csv")
broad_pathways = set(catalog.loc[catalog["pathway_level"] <= 1, "reactome_id"])
YEAR_FILL = df_train1["start_year"].median()
FP_COLS = [f"fp{i}" for i in range(1024)]
STRUCT = ["n_drugs", "n_diseases", "n_icd", "log_crit_len", "n_bullets", "has_placebo", "has_match", "n_match", "year_missing"]
CHEM = ["mol_wt", "mol_logp"]


def parse_list(s):
    try:
        return [str(x) for x in ast.literal_eval(s)]
    except (ValueError, SyntaxError):
        return []


def parse_icd(s):
    # icdcodes is a list of strings that each hold another list
    return [code for inner in parse_list(s) for code in parse_list(inner)]


def split_semicolons(s):
    return [] if pd.isna(s) else s.split(";")


def split_criteria(text):
    match = re.search(r"exclusion criteria", text, flags=re.I)
    return (text[: match.start()], text[match.start():]) if match else (text, "")


def underscore(items):
    return " ".join(x.replace(" ", "_") for x in items)


def engineer(df):
    out = pd.DataFrame(index=df.index)
    criteria = df["criteria"].fillna("")
    sections = criteria.map(split_criteria)
    diseases = df["diseases_cleaned"].map(parse_list)
    drugs = df["drugs"].map(parse_list)
    icd = df["icdcodes"].map(parse_icd)

    out["phase"] = df["phase"]
    out["phase_detail"] = df["phase_detail"]
    out["criteria"] = criteria
    out["inclusion"] = sections.map(lambda x: x[0])
    out["exclusion"] = sections.map(lambda x: x[1])
    out["diseases"] = diseases.map(" ".join)
    out["drugs_txt"] = drugs.map(underscore)
    out["icd3"] = icd.map(lambda codes: " ".join(sorted({c[:3] for c in codes})))
    out["targets"] = df["drug_targets"].map(lambda s: " ".join(split_semicolons(s)))
    out["actions"] = df["drug_actions"].map(lambda s: underscore(split_semicolons(s)))
    out["pathways"] = df["drug_pathways"].map(lambda s: " ".join(p for p in split_semicolons(s) if p in broad_pathways))

    out["n_drugs"] = drugs.map(len)
    out["n_diseases"] = diseases.map(len)
    out["n_icd"] = icd.map(len)
    out["log_crit_len"] = np.log1p(criteria.str.len())
    out["n_bullets"] = criteria.str.count(r"\n\s*-")
    out["has_placebo"] = drugs.map(lambda names: any("placebo" in n.lower() for n in names)).astype(int)
    out["has_match"] = df["matched_drugs"].notna().astype(int)
    out["n_match"] = df["matched_drugs"].map(lambda s: len(split_semicolons(s)))
    out["year_missing"] = df["start_year"].isna().astype(int)
    out["year"] = df["start_year"].fillna(YEAR_FILL) - YEAR_FILL

    # Combine drugs with an element-wise max, and use zeros for trials with no matched drug
    fingerprints, weights, logps = [], [], []
    for smiles in df["drug_smiles"]:
        mols = [m for s in split_semicolons(smiles) if (m := Chem.MolFromSmiles(s)) is not None]
        if mols:
            fingerprints.append(np.max([fp_generator.GetFingerprintAsNumPy(m) for m in mols], axis=0))
            weights.append(np.mean([Descriptors.MolWt(m) for m in mols]))
            logps.append(np.mean([Descriptors.MolLogP(m) for m in mols]))
        else:
            fingerprints.append(np.zeros(1024, dtype=np.int8))
            weights.append(0.0)
            logps.append(0.0)
    out["mol_wt"], out["mol_logp"] = weights, logps
    fp_frame = pd.DataFrame(np.vstack(fingerprints), index=df.index, columns=FP_COLS).astype(np.int8)
    return pd.concat([out, fp_frame], axis=1)


X_train, X_val, X_test = engineer(df_train1), engineer(df_valid1), engineer(df_test1)

# Guard against leaking identifiers or reason information into features
forbidden = {"ID", "trial_id", "label", "Safety", "Funding", "Efficacy", "Recruitment", "Other", "source"}
assert not forbidden & set(X_train.columns) and not any(c.startswith("emb_") for c in X_train.columns)
print(X_train.shape, X_val.shape, X_test.shape)

### Evaluation harness

`fit_predict(A, y, B)` is the interface for every model: train on frame `A` with labels `y`, return probabilities for frame `B`.

In [ ]:
THRESHOLD_GRID = np.arange(0.05, 0.95, 0.005)


def f1_fast(y, predicted):
    tp = np.sum(predicted & (y == 1))
    denominator = predicted.sum() + (y == 1).sum()
    return 2 * tp / denominator if denominator else 0.0


def best_threshold(y, p):
    return float(THRESHOLD_GRID[np.argmax([f1_fast(y, p >= t) for t in THRESHOLD_GRID])])


def cross_predict(fit_predict):
    # Out-of-fold, validation and time-holdout predictions for one model
    oof = np.zeros(len(y_train))
    for a, b in GroupKFold(5).split(X_train, y_train, groups_train):
        assert not set(groups_train[a]) & set(groups_train[b])
        oof[b] = fit_predict(X_train.iloc[a], y_train[a], X_train.iloc[b])
    val = fit_predict(X_train, y_train, X_val)
    time = fit_predict(X_train[OLD], y_train[OLD], X_train[NEW])
    return {"oof": oof, "val": val, "time": time}


OLD = df_train1["start_year"].values <= 2009
NEW = df_train1["start_year"].values >= 2010


def thresholds_from_oof(preds):
    return {ph: best_threshold(y_train[phase_train == ph], preds["oof"][phase_train == ph]) for ph in PHASES}


def evaluate_phase(name, preds):
    # One row per phase. The threshold comes from train OOF and is applied unchanged to validation.
    rows = []
    thresholds = thresholds_from_oof(preds)
    phase_new, y_new = phase_train[NEW], y_train[NEW]
    for ph in PHASES:
        k, v, n = phase_train == ph, phase_val == ph, phase_new == ph
        t = thresholds[ph]
        y, p = y_val[v], preds["val"][v]
        rows.append({
            "model": name, "phase": ph, "threshold": t,
            "cv_f1": f1_fast(y_train[k], preds["oof"][k] >= t),
            "cv_auc": roc_auc_score(y_train[k], preds["oof"][k]),
            "cv_logloss": log_loss(y_train[k], preds["oof"][k]),
            "val_f1": f1_score(y, p >= t),
            "val_precision": precision_score(y, p >= t, zero_division=0),
            "val_recall": recall_score(y, p >= t),
            "val_auc": roc_auc_score(y, p),
            "val_pr_auc": average_precision_score(y, p),
            "val_logloss": log_loss(y, p),
            "time_auc": roc_auc_score(y_new[n], preds["time"][n]),
            "time_logloss": log_loss(y_new[n], preds["time"][n]),
            "time_bias": preds["time"][n].mean() - y_new[n].mean(),
        })
    return rows


MEAN_COLS = ["cv_f1", "cv_auc", "cv_logloss", "val_f1", "val_auc", "val_pr_auc", "val_logloss", "time_auc", "time_logloss", "time_bias"]


def mean_table(rows):
    return pd.DataFrame(rows).groupby("model", sort=False)[MEAN_COLS].mean().round(3)


# Reference points every model must beat
rows = []
for ph, frame in {"I": df_train_I, "II": df_train_II, "III": df_train_III}.items():
    v = phase_val == ph
    rows.append({
        "phase": ph, "train_rows": len(frame), "train_success_rate": frame["label"].mean(), "val_rows": v.sum(),
        "all_positive_f1": f1_score(y_val[v], np.ones(v.sum())),
        "constant_rate_logloss": log_loss(y_val[v], np.full(v.sum(), frame["label"].mean())),
    })
reference = pd.DataFrame(rows).set_index("phase")
reference.round(3)

In [ ]:
# Success rate climbs with start year, and every test trial starts after the training data ends
year_bins = pd.cut(df_train1["start_year"], [0, 2000, 2005, 2008, 2010, 2013])
display(df_train1.groupby(["phase", year_bins], observed=True)["label"].mean().unstack().round(3))
print("Train years:", df_train1["start_year"].min(), "to", df_train1["start_year"].max())
print("Test years: ", df_test1["start_year"].min(), "to", df_test1["start_year"].max())

### Feature blocks

Add one block at a time to logistic regression, with `C` tuned by CV log loss. The last row repeats the full feature set without scaling the blocks, to show how much block scale matters. Every block is unit-scale otherwise.

In [ ]:
CS = [0.1, 0.3, 1, 3, 10, 30]


def tfidf(ngram=(1, 2), min_df=3):
    return TfidfVectorizer(ngram_range=ngram, min_df=min_df, max_features=100000, sublinear_tf=True)


def token_tfidf():
    return TfidfVectorizer(token_pattern=r"\S+", lowercase=False, min_df=3, sublinear_tf=True)


def token_binary():
    return CountVectorizer(token_pattern=r"\S+", lowercase=False, binary=True, min_df=3)


def fit_blocks(names, A, B, scaled=True):
    # Fit each named block on A. Returns sparse matrices for A and B.
    token = token_tfidf if scaled else token_binary
    text_blocks = {
        "crit": (tfidf, "criteria"), "incl": (tfidf, "inclusion"), "excl": (tfidf, "exclusion"),
        "dis": (lambda: tfidf(min_df=2), "diseases"), "drug": (token, "drugs_txt"), "icd": (token, "icd3"),
        "targets": (token, "targets"), "actions": (token, "actions"), "pathways": (token, "pathways"),
    }
    out_a, out_b = [], []
    for name in names:
        if name in text_blocks:
            make, column = text_blocks[name]
            vec = make()
            out_a.append(vec.fit_transform(A[column]))
            out_b.append(vec.transform(B[column]))
    if "detail" in names:
        enc = OneHotEncoder(handle_unknown="ignore").fit(A[["phase_detail"]])
        out_a.append(enc.transform(A[["phase_detail"]]))
        out_b.append(enc.transform(B[["phase_detail"]]))
    numeric = (STRUCT if "struct" in names else []) + (CHEM if "chem" in names else [])
    if numeric:
        scaler = StandardScaler().fit(A[numeric])
        weight = 1 / np.sqrt(len(numeric)) if scaled else 1.0
        out_a.append(sparse.csr_matrix(scaler.transform(A[numeric]) * weight))
        out_b.append(sparse.csr_matrix(scaler.transform(B[numeric]) * weight))
    if "fp" in names:
        fa, fb = sparse.csr_matrix(A[FP_COLS].values.astype(float)), sparse.csr_matrix(B[FP_COLS].values.astype(float))
        out_a.append(normalize(fa) if scaled else fa)
        out_b.append(normalize(fb) if scaled else fb)
    return sparse.hstack(out_a).tocsr(), sparse.hstack(out_b).tocsr()


def augment(X, phase):
    # Pooled model: a shared copy of the features, one copy per phase, and a phase intercept
    parts = [X] + [sparse.diags((phase == ph).astype(float)) @ X for ph in PHASES]
    parts.append(sparse.csr_matrix(np.column_stack([(phase == ph).astype(float) for ph in PHASES])))
    return sparse.hstack(parts).tocsr()


def logistic(C):
    return LogisticRegression(C=C, max_iter=3000, solver="liblinear")


def block_cv(label, names, mode, Cs=CS, scaled=True):
    # mode is a phase name for a per-phase model, or "pooled"
    mask = np.ones(len(y_train), bool) if mode == "pooled" else phase_train == mode
    A, y, g, phases = X_train[mask], y_train[mask], groups_train[mask], phase_train[mask]
    oof = {C: np.zeros(len(y)) for C in Cs}
    for a, b in GroupKFold(5).split(A, y, g):
        assert not set(g[a]) & set(g[b])
        Xa, Xb = fit_blocks(names, A.iloc[a], A.iloc[b], scaled)
        if mode == "pooled":
            Xa, Xb = augment(Xa, phases[a]), augment(Xb, phases[b])
        for C in Cs:
            oof[C][b] = logistic(C).fit(Xa, y[a]).predict_proba(Xb)[:, 1]
    rows = []
    for C in Cs:
        for ph in PHASES if mode == "pooled" else [mode]:
            k = phases == ph
            t = best_threshold(y[k], oof[C][k])
            rows.append({
                "features": label, "mode": "pooled" if mode == "pooled" else "per-phase", "C": C, "phase": ph, "threshold": t,
                "f1": f1_fast(y[k], oof[C][k] >= t), "auc": roc_auc_score(y[k], oof[C][k]),
                "pr_auc": average_precision_score(y[k], oof[C][k]), "logloss": log_loss(y[k], oof[C][k]),
            })
    return rows


ALL_BLOCKS = ["crit", "dis", "drug", "icd", "detail", "struct", "targets", "actions", "pathways", "chem", "fp"]
BLOCK_CONFIGS = {
    "criteria": ["crit"],
    "inclusion + exclusion": ["incl", "excl"],
    "+ diseases": ALL_BLOCKS[:2],
    "+ drug names": ALL_BLOCKS[:3],
    "+ ICD": ALL_BLOCKS[:4],
    "+ phase detail, counts": ALL_BLOCKS[:6],
    "+ targets, actions": ALL_BLOCKS[:8],
    "+ pathways": ALL_BLOCKS[:9],
    "+ chemistry, fingerprints": ALL_BLOCKS,
}
tasks = [(label, names, mode) for label, names in BLOCK_CONFIGS.items() for mode in PHASES + ["pooled"]]
results = Parallel(n_jobs=N_JOBS)(delayed(block_cv)(*task) for task in tasks)
results += Parallel(n_jobs=3)(delayed(block_cv)("everything, unscaled, C=1", ALL_BLOCKS, ph, [1], False) for ph in PHASES)
block_results = pd.DataFrame(sum(results, []))

# Best C for each features, mode and phase combination, chosen by CV log loss
block_best = block_results.loc[block_results.groupby(["features", "mode", "phase"], sort=False)["logloss"].idxmin()]
print("Mean across phases at the best C")
display(block_best.groupby(["mode", "features"], sort=False)[["f1", "auc", "pr_auc", "logloss"]].mean().round(3))
print("By phase")
block_order = pd.MultiIndex.from_frame(block_best[["mode", "features"]].drop_duplicates())
display(block_best.pivot(index=["mode", "features"], columns="phase", values="auc").reindex(block_order).round(3))

### Text representation

Compare bag of words, n-grams and TF-IDF on criteria text with the pooled model. The winner by mean CV log loss is used from here on.

In [ ]:
def representations():
    return {
        "BoW counts": CountVectorizer(min_df=3, max_features=100000),
        "BoW binary": CountVectorizer(min_df=3, max_features=100000, binary=True),
        "1-2 gram counts": CountVectorizer(ngram_range=(1, 2), min_df=3, max_features=100000),
        "TF-IDF 1 gram": tfidf((1, 1)),
        "TF-IDF 1-2 gram": tfidf((1, 2)),
        "TF-IDF 1-3 gram": tfidf((1, 3)),
        "TF-IDF 1-2 gram, min_df=10": tfidf((1, 2), 10),
    }


def representation_cv(label):
    oof = {C: np.zeros(len(y_train)) for C in CS}
    for a, b in GroupKFold(5).split(X_train, y_train, groups_train):
        vec = representations()[label]
        # Counts are L2-normalized so C means the same thing across representations
        Xa = normalize(vec.fit_transform(X_train["criteria"].iloc[a]))
        Xb = normalize(vec.transform(X_train["criteria"].iloc[b]))
        Xa, Xb = augment(Xa, phase_train[a]), augment(Xb, phase_train[b])
        for C in CS:
            oof[C][b] = logistic(C).fit(Xa, y_train[a]).predict_proba(Xb)[:, 1]
    rows = []
    for C in CS:
        per_phase = [(y_train[phase_train == ph], oof[C][phase_train == ph]) for ph in PHASES]
        rows.append({
            "representation": label, "C": C,
            "f1": np.mean([f1_fast(y, p >= best_threshold(y, p)) for y, p in per_phase]),
            "auc": np.mean([roc_auc_score(y, p) for y, p in per_phase]),
            "logloss": np.mean([log_loss(y, p) for y, p in per_phase]),
        })
    return rows


rep_results = pd.DataFrame(sum(Parallel(n_jobs=N_JOBS)(delayed(representation_cv)(r) for r in representations()), []))
rep_best = rep_results.loc[rep_results.groupby("representation", sort=False)["logloss"].idxmin()].set_index("representation")
BEST_REP = rep_best["logloss"].idxmin()
BEST_C = float(rep_best.loc[BEST_REP, "C"])
print("Selected:", BEST_REP, "C =", BEST_C)
rep_best.round(4)

### Year handling and model classes

`year clip` holds each model's year at its own training maximum, which is what test rows would get. Recency weighting weights rows by `exp((year - latest year) / tau)`.

Per-phase Naive Bayes and KNN, plus pooled gradient boosting, give the other model classes. Trees cannot extrapolate, so boosting treats later years like the last year it saw.

In [ ]:
def lr_pooled(C=None, year=None, tau=None, extra=False):
    C = BEST_C if C is None else C

    def fit_predict(A, y, B):
        vec = representations()[BEST_REP]
        Xa, Xb = normalize(vec.fit_transform(A["criteria"])), normalize(vec.transform(B["criteria"]))
        if extra:
            Xa2, Xb2 = fit_blocks(["dis", "drug"], A, B)
            Xa, Xb = sparse.hstack([Xa, Xa2]).tocsr(), sparse.hstack([Xb, Xb2]).tocsr()
        if year:
            ya, yb = A["year"].values, B["year"].values
            if year == "clip":
                yb = np.minimum(yb, ya.max())
            mean, sd = ya.mean(), ya.std()
            Xa = sparse.hstack([Xa, sparse.csr_matrix(((ya - mean) / sd)[:, None])]).tocsr()
            Xb = sparse.hstack([Xb, sparse.csr_matrix(((yb - mean) / sd)[:, None])]).tocsr()
        weights = None
        if tau:
            weights = np.exp((A["year"].values - A["year"].values.max()) / tau)
            weights = weights * len(weights) / weights.sum()
        model = logistic(C).fit(augment(Xa, A["phase"].values), y, sample_weight=weights)
        return model.predict_proba(augment(Xb, B["phase"].values))[:, 1]
    return fit_predict


def per_phase_text(make_vectorizer, make_classifier):
    def fit_predict(A, y, B):
        p = np.full(len(B), np.nan)
        for ph in PHASES:
            a, b = A["phase"].values == ph, B["phase"].values == ph
            if b.any():
                vec = make_vectorizer()
                model = make_classifier().fit(vec.fit_transform(A["criteria"][a]), y[a])
                p[b] = model.predict_proba(vec.transform(B["criteria"][b]))[:, 1]
        return p
    return fit_predict


def counts():
    return CountVectorizer(ngram_range=(1, 2), min_df=3, max_features=100000)


def nb(alpha, make_vectorizer):
    return per_phase_text(make_vectorizer, lambda: MultinomialNB(alpha=alpha))


def knn(k):
    return per_phase_text(tfidf, lambda: KNeighborsClassifier(n_neighbors=k, metric="cosine", weights="distance", algorithm="brute"))


def boosted(text_dim=0, drug_dim=0, use_year=True, leaves=15):
    def fit_predict(A, y, B):
        numeric = STRUCT + CHEM + (["year"] if use_year else [])
        phase_columns = lambda D: np.column_stack([(D["phase"].values == ph).astype(float) for ph in PHASES])
        enc = OneHotEncoder(handle_unknown="ignore", sparse_output=False).fit(A[["phase_detail"]])
        Fa = [A[numeric].values, phase_columns(A), enc.transform(A[["phase_detail"]])]
        Fb = [B[numeric].values, phase_columns(B), enc.transform(B[["phase_detail"]])]
        if text_dim:
            vec, svd = tfidf(), TruncatedSVD(text_dim, random_state=0)
            Fa.append(svd.fit_transform(vec.fit_transform(A["criteria"])))
            Fb.append(svd.transform(vec.transform(B["criteria"])))
        if drug_dim:
            Ma, Mb = fit_blocks(["drug", "icd", "targets", "actions", "pathways"], A, B)
            svd = TruncatedSVD(drug_dim, random_state=0)
            Fa.append(svd.fit_transform(Ma))
            Fb.append(svd.transform(Mb))
        model = HistGradientBoostingClassifier(learning_rate=0.03, max_leaf_nodes=leaves, max_iter=400, l2_regularization=1.0, min_samples_leaf=30, random_state=0)
        return model.fit(np.hstack(Fa), y).predict_proba(np.hstack(Fb))[:, 1]
    return fit_predict


MODELS = {
    "LR pooled, criteria": lr_pooled(),
    "LR pooled, criteria + diseases + drugs": lr_pooled(extra=True),
    "LR pooled, year raw": lr_pooled(year="raw"),
    "LR pooled, year clip": lr_pooled(year="clip"),
    "LR pooled, recency tau=3": lr_pooled(tau=3),
    "LR pooled, recency tau=6": lr_pooled(tau=6),
    "NB counts a=0.3": nb(0.3, counts),
    "NB counts a=1": nb(1.0, counts),
    "NB TF-IDF a=0.1": nb(0.1, tfidf),
    "NB TF-IDF a=0.3": nb(0.3, tfidf),
    "NB TF-IDF a=1": nb(1.0, tfidf),
    "KNN k=10": knn(10),
    "KNN k=25": knn(25),
    "KNN k=50": knn(50),
    "KNN k=100": knn(100),
    "HGB structured": boosted(),
    "HGB structured, no year": boosted(use_year=False),
    "HGB + text SVD": boosted(text_dim=100),
    "HGB + text SVD, 7 leaves": boosted(text_dim=100, leaves=7),
    "HGB + text SVD, 31 leaves": boosted(text_dim=100, leaves=31),
    "HGB + text SVD + drug SVD": boosted(text_dim=100, drug_dim=50),
}
PREDS = dict(zip(MODELS, Parallel(n_jobs=N_JOBS)(delayed(cross_predict)(fp) for fp in MODELS.values())))
model_rows = sum((evaluate_phase(name, preds) for name, preds in PREDS.items()), [])
print("Mean across phases")
mean_table(model_rows)

In [ ]:
pd.DataFrame(model_rows).pivot(index="model", columns="phase", values=["cv_f1", "val_f1", "val_auc", "threshold"]).reindex(list(MODELS)).round(3)

### Original model and error analysis

The original model is the simple per-phase choice: logistic regression on criteria TF-IDF, with each phase's `C` set by CV log loss.

**Which reason labels are real:** `part1/part1_reason_labels.csv` has a `source` column. `n_true` counts trials in a group with true Part 1 labels, and the rest of each reason group uses Part 1 predictions. The "no recorded reason" group is trials not in Part 1.

Every Part 1 trial in val has `label=0`. Part 1 membership would be a near-perfect feature, but it is not allowed and is absent for test trials, so it is checked here and never used.

In [ ]:
original_C = block_best[(block_best["features"] == "criteria") & (block_best["mode"] == "per-phase")].set_index("phase")["C"].to_dict()
print("Per-phase C:", original_C)


def lr_per_phase(A, y, B):
    p = np.full(len(B), np.nan)
    for ph in PHASES:
        a, b = A["phase"].values == ph, B["phase"].values == ph
        if b.any():
            vec = tfidf()
            model = logistic(original_C[ph]).fit(vec.fit_transform(A["criteria"][a]), y[a])
            p[b] = model.predict_proba(vec.transform(B["criteria"][b]))[:, 1]
    return p


original_preds = cross_predict(lr_per_phase)
original_rows = evaluate_phase("Original", original_preds)
display(pd.DataFrame(original_rows).set_index("phase")[["threshold", "val_f1", "val_precision", "val_recall", "val_auc", "val_pr_auc", "val_logloss"]].round(3))

REASONS = ["Safety", "Funding", "Efficacy", "Recruitment", "Other"]
reason_labels = pd.read_csv("../part1/part1_reason_labels.csv").set_index("ID")

for name, frame in {"train": df_train1, "val": df_valid1}.items():
    in_part1 = frame["trial_id"].isin(reason_labels.index)
    print(f"{name}: {in_part1.sum()} rows in Part 1, success rate {frame.loc[in_part1, 'label'].mean():.3f}")
assert not df_test1["trial_id"].isin(reason_labels.index).any()


def error_table(preds):
    thresholds = thresholds_from_oof(preds)
    rows = []
    for ph in PHASES:
        data = df_valid1[phase_val == ph].assign(p=preds["val"][phase_val == ph])
        positives = data.loc[data["label"] == 1, "p"].values
        negatives = data[data["label"] == 0].join(reason_labels, on="trial_id")
        assert negatives["trial_id"].is_unique
        matched = negatives[negatives["source"].notna()]
        assert (matched[REASONS].sum(axis=1) >= 1).all()
        reason_groups = {r: negatives[negatives[r] == 1] for r in REASONS}
        reason_groups["No recorded reason"] = negatives[negatives["source"].isna()]
        reason_groups["All label=0"] = negatives
        assert len(matched) + len(reason_groups["No recorded reason"]) == len(negatives)
        for group, g in reason_groups.items():
            scores = np.r_[positives, g["p"].values]
            truth = np.r_[np.ones(len(positives)), np.zeros(len(g))]
            rows.append({
                "phase": ph, "group": group, "n": len(g), "n_true": int((g["source"] == "true").sum()),
                "wrongly_success": (g["p"] >= thresholds[ph]).mean() if len(g) else np.nan,
                "mean_p": g["p"].mean() if len(g) else np.nan,
                "auc_vs_successes": roc_auc_score(truth, scores) if len(g) else np.nan,
            })
    return pd.DataFrame(rows)


original_errors = error_table(original_preds)
original_errors.round(3)

### Improving the model

Candidates are equal-weight logit averages, so nothing is fit on the predictions being scored. The rule is set in advance and does not use val: highest mean CV F1 rounded to 3 decimals, with ties broken by lowest time-holdout log loss. Val is then the check on that choice.

In [ ]:
def blend(names, key):
    return expit(np.mean([logit(np.clip(PREDS[n][key], 1e-3, 1 - 1e-3)) for n in names], axis=0))


best_knn = min((n for n in PREDS if n.startswith("KNN")), key=lambda n: np.mean([r["cv_logloss"] for r in model_rows if r["model"] == n]))
LR_RECENT, HGB_TEXT, HGB_DRUG = "LR pooled, recency tau=6", "HGB + text SVD", "HGB + text SVD + drug SVD"
CANDIDATES = {
    "LR": ["LR pooled, criteria"],
    "LR recency": [LR_RECENT],
    "LR recency + HGB": [LR_RECENT, HGB_TEXT],
    "LR recency + HGB drug": [LR_RECENT, HGB_DRUG],
    "LR recency + KNN": [LR_RECENT, best_knn],
    "LR recency + HGB + KNN": [LR_RECENT, HGB_TEXT, best_knn],
    "LR recency + HGB drug + KNN": [LR_RECENT, HGB_DRUG, best_knn],
    "LR recency + HGB + KNN + NB": [LR_RECENT, HGB_TEXT, best_knn, "NB TF-IDF a=0.3"],
}
blended = {name: {key: blend(names, key) for key in ["oof", "val", "time"]} for name, names in CANDIDATES.items()}
candidate_rows = sum((evaluate_phase(name, preds) for name, preds in blended.items()), [])
candidate_table = mean_table(candidate_rows)

ranked = candidate_table.assign(key=candidate_table["cv_f1"].round(3)).sort_values(["key", "time_logloss"], ascending=[False, True])
IMPROVED = ranked.index[0]
improved_preds = blended[IMPROVED]
print("Selected:", IMPROVED, CANDIDATES[IMPROVED])
candidate_table

In [ ]:
improved_rows = evaluate_phase("Improved", improved_preds)
comparison = pd.DataFrame(original_rows + improved_rows)
display(comparison.pivot(index="phase", columns="model", values=["threshold", "val_f1", "val_precision", "val_recall", "val_auc", "val_pr_auc", "val_logloss"]).round(3))
print("Mean across phases")
display(comparison.groupby("model", sort=False)[MEAN_COLS].mean().round(3))
print("All-positive F1 and constant-rate log loss on val")
display(reference[["all_positive_f1", "constant_rate_logloss"]].round(3))

# Paired bootstrap over val rows, with thresholds held fixed
rng = np.random.default_rng(0)
original_t, improved_t = thresholds_from_oof(original_preds), thresholds_from_oof(improved_preds)
rows = []
for ph in PHASES:
    v = phase_val == ph
    y, p0, p1 = y_val[v], original_preds["val"][v], improved_preds["val"][v]
    draws = []
    for _ in range(2000):
        i = rng.integers(0, len(y), len(y))
        draws.append((f1_fast(y[i], p0[i] >= original_t[ph]), f1_fast(y[i], p1[i] >= improved_t[ph]),
                      roc_auc_score(y[i], p0[i]), roc_auc_score(y[i], p1[i])))
    draws = np.array(draws)
    ci = lambda x: f"[{np.percentile(x, 2.5):+.3f}, {np.percentile(x, 97.5):+.3f}]"
    rows.append({
        "phase": ph, "n": len(y),
        "improved_f1": f1_fast(y, p1 >= improved_t[ph]),
        "improved_f1_95ci": f"[{np.percentile(draws[:, 1], 2.5):.3f}, {np.percentile(draws[:, 1], 97.5):.3f}]",
        "f1_change_95ci": ci(draws[:, 1] - draws[:, 0]),
        "auc_change_95ci": ci(draws[:, 3] - draws[:, 2]),
    })
pd.DataFrame(rows).set_index("phase").round(3)

In [ ]:
improved_errors = error_table(improved_preds)
error_comparison = original_errors.merge(improved_errors, on=["phase", "group", "n", "n_true"], suffixes=("_original", "_improved"))
error_comparison[["phase", "group", "n", "n_true", "wrongly_success_original", "wrongly_success_improved", "auc_vs_successes_original", "auc_vs_successes_improved"]].round(3)

### Final fit and submission

Refit the selected models on train + val with the settings chosen above, then write probabilities for Kaggle.

In [ ]:
X_all = pd.concat([X_train, X_val], ignore_index=True)
y_all = np.r_[y_train, y_val]

test_preds = Parallel(n_jobs=N_JOBS)(delayed(MODELS[name])(X_all, y_all, X_test) for name in CANDIDATES[IMPROVED])
test_probability = expit(np.mean([logit(np.clip(p, 1e-3, 1 - 1e-3)) for p in test_preds], axis=0))

submission = pd.DataFrame({"ID": df_test1["ID"], "label": test_probability})
sample = pd.read_csv("part2_sample_submission.csv")

assert len(submission) == 1500 and list(submission.columns) == list(sample.columns)
assert set(submission["ID"]) == set(sample["ID"]) and submission["ID"].is_unique
assert submission["label"].notna().all() and submission["label"].between(0, 1, inclusive="neither").all()

submission.to_csv("part2_submission.csv", index=False)
print(submission.assign(phase=df_test1["phase"]).groupby("phase")["label"].describe().round(3))